# Exercise 2 · Understand the agent loop — round by round
**ETH MAS 2026 · Module 1 · B2 — From LLMs to agents**

In Exercise 1 *your code* decided: search once, then answer. Now the **LLM decides** what to search and how often. Your code runs a **loop**:

```
          ┌──────────────────────────────────────────────────┐
question ─►  LLM (+ tool schema)                              │
          │    ├─ tool call (filled-in JSON) ──► your code runs the search ─┘
          │    └─ no tool call ──► final answer
```

You run the loop **one round per cell** and look at what goes **in** and what comes **out**. No coding needed for the main part.

**The question needs a hop:** the first search tells you *which* company Siemens bought — only then can the agent search for how *that* company developed.

**Time:** ~35 min · alone or in a group, as you like

## 0 · Setup (run once)

1. **File → Save a copy in Drive** — otherwise your changes are lost.
2. Run the next three cells. The last one should print a short greeting from the model.

The course API key is already in the notebook (see the cell below). *Optional, the clean way:* store it under the **key icon** in the left sidebar (*Secrets*) as `UNIQUE_API_KEY` — the notebook uses that first.

> After a break, Colab may have reset: use **Runtime → Run all** (or run from the top again).

In [ ]:
!pip install -q unique-sdk

In [5]:
import json
import re
import unique_sdk

# --- API key -------------------------------------------------------------------
# COURSE KEY: only for this course, revoked after Module 1 (21 Nov 2026).
# Never put real keys into a notebook — in production, keys live in a secret store.
COURSE_API_KEY = "ukey_4NzGvG0FxuZipyMzJdj6c7ftIObvRWt95ZwG8FnEfGQ"

try:  # the clean way: Colab Secrets (key icon in the left sidebar), if you set one
    from google.colab import userdata
    API_KEY = userdata.get("UNIQUE_API_KEY") or COURSE_API_KEY
except Exception:
    API_KEY = COURSE_API_KEY

# --- Course environment (provided by the lecturers) ---
unique_sdk.api_base = "https://gateway.unique.app/public/chat"
unique_sdk.api_key = API_KEY
unique_sdk.app_id = "app_eyugp8qq242fzp1lle4xjqc1"

COMPANY_ID = "346686658780860534"
USER_ID = "399599116591567964"
SCOPE_IDS = ["scope_nnqbp55xfh6s25wx4mdndff0"]   # knowledge base: company annual reports
MODEL = "AZURE_GPT_51_2025_1113"

In [7]:
# Connection check: one plain LLM call
hello = unique_sdk.ChatCompletion.create(
    company_id=COMPANY_ID,
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello to the ETH MAS class in one short sentence."},
    ],
)
print(hello.choices[0].message.content)

Hello ETH MAS class—wishing you an inspiring and successful journey through your studies!


## 1 · The tool — what the model sees
The model never runs code. It only sees this **schema** and answers with a *request* to call the tool, filled in as JSON.

**Reason first:** `reason` comes before `search_string`. The model writes its arguments in that order, token by token — so it first states what it knows and what is missing, then picks the query.

In [8]:
def search_knowledge_base(reason: str, search_string: str) -> str:
    """The function your code runs when the model asks for the tool.
    `reason` is not used by the search — it makes the model think before it acts, and lets you see why."""
    chunks = unique_sdk.Search.create(
        user_id=USER_ID,
        company_id=COMPANY_ID,
        searchString=search_string,
        searchType="VECTOR",
        scopeIds=SCOPE_IDS,
        limit=4,   # deliberately small: the agent has to search several times
    ).data
    return "\n\n".join(
        f"[{c.get('title') or c.get('key')}, p. {c.get('startPage')}–{c.get('endPage')}]\n{c['text']}"
        for c in chunks
    )


# The tool schema: this is ALL the model sees of your tool
search_tool = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Search the internal knowledge base of annual reports (Siemens, ABB, UBS; 2021–2023). "
                       "Returns a few text passages with document title and page. "
                       "Use short, specific queries (one topic, one year, one company).",
        "parameters": {
            "type": "object",
            "properties": {
                "reason": {   # FIRST: the model writes its reasoning before it picks the query
                    "type": "string",
                    "description": "Why you search this now: what you already know and what is still missing.",
                },
                "search_string": {
                    "type": "string",
                    "description": "The search query, e.g. 'Siemens revenue fiscal year 2021'",
                },
            },
            "required": ["reason", "search_string"],
        },
    },
}

TOOLS = {"search_knowledge_base": search_knowledge_base}   # name -> Python function

In [9]:
print(json.dumps(search_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "search_knowledge_base",
    "description": "Search the internal knowledge base of annual reports (Siemens, ABB, UBS; 2021\u20132023). Returns a few text passages with document title and page. Use short, specific queries (one topic, one year, one company).",
    "parameters": {
      "type": "object",
      "properties": {
        "reason": {
          "type": "string",
          "description": "Why you search this now: what you already know and what is still missing."
        },
        "search_string": {
          "type": "string",
          "description": "The search query, e.g. 'Siemens revenue fiscal year 2021'"
        }
      },
      "required": [
        "reason",
        "search_string"
      ]
    }
  }
}


> **Look at the output:**
> - Which fields must the model fill in? Which one comes first — and why does that matter?

## 2 · The agent's instructions and the question

In [10]:
AGENT_SYSTEM_PROMPT = """You are a research analyst with access to a search tool over company annual reports.
Work step by step:
1. Break the question into sub-questions (per company, year and topic).
2. Search ONE sub-question per turn with a short, specific query. Read the results, then decide the next search.
   Use different queries — never repeat one.
3. Do at least 3 search rounds before you answer.
4. When you have enough evidence, write the final answer: structured, concise, and cite the
   document and page for every number, e.g. (Siemens Annual Report 2021, p. 12).
"""

QUESTION = ("In fiscal 2021 Siemens completed a large acquisition. Which company was it, "
            "what did Siemens pay, and how did that business develop in fiscal 2022 and 2023?")

# Alternative (no hop, but several searches): "Compare the 2021 and 2023 annual reports of Siemens: revenue, profit and strategic priorities."

## 3 · Run the loop — one cell per round
Run the next cell once (it prepares the conversation), then the round cells one after another.

In [11]:
messages = [
    {"role": "system", "content": AGENT_SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION},
]
round_no = 0
finished = False


def next_round():
    """One loop iteration: ask the model -> tool call? run it : final answer."""
    global round_no, finished
    if finished:
        print("The agent has already finished — see the final answer above.")
        return
    round_no += 1
    print(f"ROUND {round_no} · sending {len(messages)} messages to the model …\n")

    response = unique_sdk.ChatCompletion.create(
        company_id=COMPANY_ID, model=MODEL, messages=messages, options={"tools": [search_tool]},
    )
    message = response.choices[0].message
    tool_calls = message.get("toolCalls") or message.get("tool_calls") or []

    if not tool_calls:
        finished = True
        messages.append({"role": "assistant", "content": message.get("content") or ""})
        print("MODEL RESPONSE:  toolCalls = None")
        print("➜ No tool call: the MODEL decided it has enough evidence. This is the final answer.\n")
        print("=" * 80)
        print(message.get("content"))
        return

    # The model did not answer — it returned the tool schema, filled in:
    print("MODEL RESPONSE:  content = " + repr(message.get("content") or None) + ",  toolCalls =")
    print(json.dumps(
        [{"id": c["id"], "type": c.get("type", "function"),
          "function": {"name": c["function"]["name"], "arguments": json.loads(c["function"]["arguments"])}}
         for c in tool_calls],
        indent=2, ensure_ascii=False))

    messages.append({"role": "assistant", "content": message.get("content") or "", "toolCalls": tool_calls})
    for call in tool_calls:
        name = call["function"]["name"]
        args = json.loads(call["function"]["arguments"])
        result = TOOLS[name](**args)                          # YOUR code executes the call
        messages.append({"role": "tool", "toolCallId": call["id"], "name": name, "content": result})
        sources = re.findall(r"^\[(.+?)\]$", result, flags=re.M)
        print(f"\nYOUR CODE ran {name}({args})")
        print(f"  → {len(sources)} passages sent back to the model: " + "; ".join(sources))

### Round 1

In [12]:
next_round()

ROUND 1 · sending 2 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_vrFv6DJnfaqlO4u8WFa3h1hc",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "Identify major acquisition by Siemens in fiscal 2021 and purchase price",
        "search_string": "Siemens major acquisition 2021 purchase price"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'Identify major acquisition by Siemens in fiscal 2021 and purchase price', 'search_string': 'Siemens major acquisition 2021 purchase price'})
  → 4 passages sent back to the model: Siemens_2022.pdf, p. 51–51; Siemens_2021.pdf, p. 53–53; Siemens_2022.pdf, p. 51–51; Siemens_2021.pdf, p. 53–53


> **Look at the output:**
> - The model returned **no text**, but `toolCalls`. Read the `reason`: what does the model know so far?
> - Which query did it choose? Which passages did your code send back?

### Round 2

In [13]:
next_round()

ROUND 2 · sending 4 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_lRIFtv5jRSsefYhOuh6tRSer",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "Clarify which acquisition is described as the large 2021 deal – likely Varian by Siemens Healthineers – and its purchase price",
        "search_string": "Varian Medical Systems acquisition purchase price Siemens fiscal 2021"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'Clarify which acquisition is described as the large 2021 deal – likely Varian by Siemens Healthineers – and its purchase price', 'search_string': 'Varian Medical Systems acquisition purchase price Siemens fiscal 2021'})
  → 4 passages sent back to the model: Siemens_2022.pdf, p. 51–51; Siemens_2021.pdf, p. 52–53; Siemens_2021.pdf, p. 105–106; Siemens_2021.pdf, p. 53–53


> **Look at the output:**
> - Compare the query with round 1: does it use something the model **only learned from the result** of round 1?

### Round 3

In [14]:
next_round()

ROUND 3 · sending 6 messages to the model …

MODEL RESPONSE:  content = None,  toolCalls =
[
  {
    "id": "call_j64hltMh5IVVbOP4mkjvp5p5",
    "type": "function",
    "function": {
      "name": "search_knowledge_base",
      "arguments": {
        "reason": "Find how Varian business developed in fiscal 2022 and 2023 within Siemens/Siemens Healthineers",
        "search_string": "Varian business development fiscal 2022 2023 Siemens Healthineers"
      }
    }
  }
]

YOUR CODE ran search_knowledge_base({'reason': 'Find how Varian business developed in fiscal 2022 and 2023 within Siemens/Siemens Healthineers', 'search_string': 'Varian business development fiscal 2022 2023 Siemens Healthineers'})
  → 4 passages sent back to the model: Siemens_2022.pdf, p. 13–13; Siemens_2023.pdf, p. 13–13; Siemens_2022.pdf, p. 12–13; Siemens_2021.pdf, p. 13–14


> **Look at the output:**
> - What is still missing according to the `reason`?

### Round 4

In [15]:
next_round()

ROUND 4 · sending 8 messages to the model …

MODEL RESPONSE:  toolCalls = None
➜ No tool call: the MODEL decided it has enough evidence. This is the final answer.

**Largest acquisition in fiscal 2021**

In fiscal 2021, Siemens (via its majority-owned subsidiary Siemens Healthineers AG) completed the acquisition of **Varian Medical Systems, Inc.** (Varian), a U.S. company active in cancer care, especially radiation oncology and related digital solutions:

- **Target:** Varian Medical Systems, Inc., USA  
- **Closing date:** April 15, 2021  
- **Cash purchase price:** **US$16.4 billion**, equivalent to **€13.9 billion** at the acquisition date (Siemens Annual Report 2021, Note 3, p. 52–53; Combined Management Report, p. 13–14).

The preliminary purchase price allocation included goodwill of €8.0 billion and other intangible assets of €6.3 billion (Siemens Annual Report 2021, Note 3, p. 52–53). Varian contributed **€1,241 million in revenue** and a **net loss of €50 million** from the ac

> **Look at the output:**
> - If you see `toolCalls = None`: **who** decided that this is the final answer — your code or the model?
> - Check two numbers in the answer against the pages the searches returned.

### Round 5 (only if the agent is still searching — if needed, run `next_round()` again)

In [ ]:
next_round()

## 4 · What the model saw
The agent has no hidden memory: every round re-sends this growing list.

In [16]:
# The whole conversation the model saw — the agent's "memory" is just this list
for i, m in enumerate(messages, 1):
    role = m["role"]
    if role == "assistant" and m.get("toolCalls"):
        calls = ", ".join(f'{c["function"]["name"]}({c["function"]["arguments"]})' for c in m["toolCalls"])
        text = "→ tool call: " + calls
    elif role == "tool":
        n = len(re.findall(r"^\[", m["content"], flags=re.M))
        text = f"{n} passages ({len(m['content']):,} chars)"
    elif role == "assistant":
        text = f"FINAL ANSWER ({len(m['content']):,} chars)"
    else:
        text = m["content"].strip().splitlines()[0][:90]
    print(f"{i:>2}  {role:<9}  {text}")

 1  system     You are a research analyst with access to a search tool over company annual reports.
 2  user       In fiscal 2021 Siemens completed a large acquisition. Which company was it, what did Sieme
 3  assistant  → tool call: search_knowledge_base({"reason":"Identify major acquisition by Siemens in fiscal 2021 and purchase price","search_string":"Siemens major acquisition 2021 purchase price"})
 4  tool       4 passages (11,322 chars)
 5  assistant  → tool call: search_knowledge_base({"reason":"Clarify which acquisition is described as the large 2021 deal – likely Varian by Siemens Healthineers – and its purchase price","search_string":"Varian Medical Systems acquisition purchase price Siemens fiscal 2021"})
 6  tool       4 passages (12,404 chars)
 7  assistant  → tool call: search_knowledge_base({"reason":"Find how Varian business developed in fiscal 2022 and 2023 within Siemens/Siemens Healthineers","search_string":"Varian business development fiscal 2022 2023 Siemens Health

> **Look at the output:**
> - How many messages did the last call send? How many characters? What does that mean for cost and speed?

## 5 · The same loop as code — about 20 lines
This function does exactly what you just did by hand. Read it and find:
1. the line where the **model decides** (tool call or answer),
2. the line where **your code executes** the tool,
3. the line that **stops** the loop.

In [20]:
def run_agent(question: str, max_steps: int = 10, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = unique_sdk.ChatCompletion.create(
            company_id=COMPANY_ID, model=MODEL, messages=messages, options={"tools": [search_tool]},
        )
        message = response.choices[0].message
        tool_calls = message.get("toolCalls") or message.get("tool_calls") or []

        if not tool_calls:                                   # no tool call = the MODEL decided: final answer
            if verbose:
                print(f"Step {step} · no tool call → final answer\n" + "=" * 60)
            return message.get("content") or ""

        messages.append({"role": "assistant", "content": message.get("content") or "", "toolCalls": tool_calls})
        for call in tool_calls:
            name = call["function"]["name"]
            args = json.loads(call["function"]["arguments"])
            result = TOOLS[name](**args)                     # the model proposes, YOUR code executes
            if verbose:
                print(f"Step {step} · {name}({args})")
            messages.append({"role": "tool", "toolCallId": call["id"], "name": name, "content": result})

    return "Stopped: max_steps reached without a final answer."   # safety limit, not a decision

In [19]:
answer = run_agent(QUESTION)
print(answer)

Step 1 · search_knowledge_base({'reason': 'Identify major acquisition Siemens completed in fiscal 2021 and purchase price.', 'search_string': 'Siemens major acquisition 2021 purchase price'})
Step 2 · search_knowledge_base({'reason': 'Confirm which acquisition is described as large/major and get more detail on Varian acquisition timings and price.', 'search_string': 'Siemens Varian acquisition purchase price 2021'})
Step 3 · search_knowledge_base({'reason': 'Find how Varian business developed in fiscal 2022 at Siemens Healthineers level: revenue, profit, margin.', 'search_string': 'Siemens Healthineers Varian segment performance 2022'})
Step 4 · no tool call → final answer
**1. Which company did Siemens acquire in fiscal 2021?**

- On April 15, 2021, Siemens (via Siemens Healthineers) completed the acquisition of **Varian Medical Systems, Inc., USA (Varian)**, a cancer‑care company focused on radiation oncology and related digital solutions and applications (Siemens Annual Report 2021,

> **Look at the output:**
> - Run it twice: are the queries the same? Is the answer the same?

## 6 · Experiments
1. **Who decides when to stop?** Delete rule 3 ("at least 3 search rounds") from `AGENT_SYSTEM_PROMPT`, run the cell again, then `run_agent(QUESTION)`. How many rounds now?
2. **The schema steers the model.** Change the `description` of the tool or of `reason` and watch the queries change.
3. **Your own question** about Siemens, ABB or UBS (2021–2023).

---
## Optional — build the loop yourself
For those who like to code: fill in the four `TODO`s, then run `run_agent_own(QUESTION)`.
Hints: `call["function"]["name"]`, `json.loads(call["function"]["arguments"])`, `TOOLS[name](**args)`, and the tool message
`{"role": "tool", "toolCallId": call["id"], "name": name, "content": result}`.

In [ ]:
def run_agent_own(question: str, max_steps: int = 10, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        # 1. Ask the model what to do next: answer, or request tool calls
        response = unique_sdk.ChatCompletion.create(
            company_id=COMPANY_ID, model=MODEL, messages=messages, options={"tools": [search_tool]},
        )
        message = response.choices[0].message
        tool_calls = message.get("toolCalls") or message.get("tool_calls") or []

        # 2. No tool call -> the model decided it has enough evidence
        if not tool_calls:
            # TODO 4: return the model's answer (message content)
            ...

        # 3. Keep the model's request in the history, then execute each tool call
        messages.append({"role": "assistant", "content": message.get("content") or "", "toolCalls": tool_calls})
        for call in tool_calls:
            # TODO 1: read the tool name and parse the JSON arguments of `call`
            name = ...
            args = ...

            # TODO 2: run the matching Python function from TOOLS
            result = ...
            if verbose:
                print(f"Step {step} · {name}({args})")

            # TODO 3: append the result as a "tool" message (role, toolCallId, name, content)
            messages.append(...)

    return "Stopped: max_steps reached without a final answer."

In [ ]:
# print(run_agent_own(QUESTION))


## Take-away
- An agent is **an LLM call in a loop**: the model proposes tool calls as JSON, **your code executes** them and appends the result.
- The **model decides** when to stop: a response without a tool call is the final answer. Your code only sets limits (`max_steps`, allowed tools).
- The model has no memory — only the **message list** that your code re-sends every round.